# AG01 · What Is an Agent? Workflows vs Agents

**Track:** Agents (lesson 1 of 15) · **Time:** ~2 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** A01 (LLM APIs) · **You will build:** an autonomy-ladder *decision helper* with a test-suite, and a side-by-side workflow vs agent

---
## 🎯 Why this matters
"Agent" is the most overloaded word in AI. Teams that skip the definition build a 12-step autonomous system for a task that a
single prompt (or a 20-line `if` statement) would have solved more cheaply, more reliably and more auditably. Teams that
swing the other way hard-code a brittle pipeline for a problem whose path is genuinely unpredictable.
The most valuable skill in this track is not *building* agents. It is **deciding when not to**, and how much autonomy to grant when you do.

**Real scenario.** You are the AI lead at a mid-size NZ online retailer. Six teams each bring you a request ("automate returns",
"auto-answer invoice queries", "let AI approve small refunds"…). You have budget for two projects. Which are agents, which are
workflows, which are a single prompt, and which should be declined *for now*? You need a defensible rubric, not a gut feeling.

## 🧭 Learning objectives
By the end you can:
1. Define *workflow* vs *agent* precisely, and place any system on the autonomy ladder (L0–L4).
2. Apply the **4-question test** (complexity, value, viability, cost of error) to a real task and justify the level you pick.
3. Implement and unit-test a decision helper that is monotone in risk (more error cost never means more autonomy).
4. Show with a trace why a workflow has a fixed number of model calls and an agent does not.
5. Quantify two structural failure modes of agents: compounding per-step error and context-cost growth.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept

### Workflow vs agent
The distinction used throughout this course (it follows Anthropic's *Building effective agents*):

* **Workflow**: LLMs and tools are orchestrated through **predefined code paths**. *Your code decides what happens next.*
* **Agent**: the LLM **dynamically directs its own process and tool use**, deciding at each step what to do next, until it judges the job done
  (or a guard stops it). *The model decides what happens next.*

A useful one-line test: **who owns the control flow?** If you can draw the whole flow chart before seeing the input, it is a workflow.
If the flow chart depends on what the model finds along the way, it is an agent.

### The autonomy ladder
Autonomy is a dial, not a switch. We use five rungs:

| Level | Name | Who owns control flow | Example | Typical cost / latency |
|---|---|---|---|---|
| **L0** | Single call | you (one call) | Summarise a meeting transcript | 1 call |
| **L1** | Fixed workflow | you (chain / route / parallel in code) | Classify ticket → extract fields → draft reply | N calls, N known in advance |
| **L2** | Bounded agent | model, inside a tight sandbox (read-only tools, few turns, human approves writes) | Order-support agent that looks things up | Variable, capped by `max_turns` |
| **L3** | Autonomous agent | model, with write tools and guardrails | Coding agent that edits files and runs tests | Variable, high |
| **L4** | Open-ended / long-horizon | model, possibly several models, hours of work | Deep-research agent producing a cited report | Highest, needs strong evals |

**Climb only as far as the task forces you to.** Every rung up buys flexibility and costs predictability, latency, money and debuggability.

### The 4-question test
Before granting a task more than L1, answer four questions (score each 0-3, our rubric for the lab):

| Question | Asks | Score 0 → 3 |
|---|---|---|
| **Complexity** | Is the path unpredictable? | one step → fixed steps → some branching → open-ended exploration |
| **Value** | Is the outcome worth a variable, higher cost per run? | trivial → modest → high → very high |
| **Viability** | Do the tools, data and model capability exist *today*? | missing → shaky → mostly OK → proven |
| **Cost of error** | What does a wrong action cost, and can it be undone? | harmless → cheap to reverse → costly → irreversible/regulated |

### Anatomy of an agent
```
           ┌──────────────── guardrails (budgets, approvals, allow/deny) ────────────────┐
  task ──▶ │  MODEL ◀──────── memory / context (messages, notes, retrieved facts) ─────┐ │
           │    │ decides                                                              │ │
           │    ▼                                                                      │ │
           │  TOOLS ──▶ observations ──────────────────────────────────────────────────┘ │
           │   (loop until done, or a guard fires)                                         │
           └───────────────────────────────────────────────────────────────────────────────┘
```
Five parts: **model**, **tools**, **memory/context**, **loop**, **guardrails**. AG02 builds the loop, AG03 the tools,
AG05 memory, AG09 the guardrails. This lesson is about the decision *before* any of that.

## 2 · Build it: a decision helper you can test
A rubric that lives in a slide deck gets argued about. A rubric that lives in code can be **unit-tested, versioned and challenged with data**.
We encode the 4-question test as a function `recommend(task) -> Recommendation`. The rules, in priority order:

1. **Viability gate.** `viability <= 1` → verdict `not_yet` (don't build; collect data / wait for capability / build the tool first).
2. **Base level from complexity.** complexity 0→L0, 1→L1, 2→L2, 3→L3.
3. **Value cap.** `value == 0` → cap at L1 (a variable-cost loop is not worth it for a trivial outcome).
4. **L4 needs everything:** only promote L3→L4 when complexity 3, value 3, viability 3 *and* error cost ≤ 1.
5. **Error-cost cap.** error cost 3 → cap at L2; error cost 2 → cap at L3. Error cost ≥ 2 also requires **human approval** on write actions.

*Predict before you run:* a task with complexity 3, value 3, viability 3 but error cost 3 (say, autonomously wiring payments).
What level should it get? (Write your guess down; the tests below will tell you whether the code agrees.)

In [ ]:
from dataclasses import dataclass, field

LADDER = {0: "L0 single call", 1: "L1 fixed workflow", 2: "L2 bounded agent", 3: "L3 autonomous agent", 4: "L4 open-ended agent"}

@dataclass
class Task:
    name: str
    complexity: int      # 0-3  how unpredictable is the path?
    value: int           # 0-3  how much is a successful run worth?
    viability: int       # 0-3  can tools/data/model do it today?
    error_cost: int      # 0-3  how bad / irreversible is a wrong action?

    def __post_init__(self):
        for f in ("complexity", "value", "viability", "error_cost"):
            v = getattr(self, f)
            if v not in (0, 1, 2, 3):
                raise ValueError(f"{f} must be an integer 0-3, got {v!r}")

@dataclass
class Recommendation:
    level: int
    verdict: str                 # "build" | "not_yet"
    human_approval: bool
    reasons: list = field(default_factory=list)

    @property
    def label(self): return LADDER[self.level]

def recommend(t: Task) -> Recommendation:
    reasons = []
    if t.viability <= 1:
        return Recommendation(0, "not_yet", False, [f"viability={t.viability}: tools/data/model not ready; do not build an agent yet"])
    level = t.complexity                      # rule 2: complexity 0..3 -> L0..L3
    reasons.append(f"complexity={t.complexity} -> start at L{level}")
    if t.value == 0 and level > 1:            # rule 3
        level = 1; reasons.append("value=0 -> capped at L1 (not worth a variable-cost loop)")
    if level == 3 and t.complexity == 3 and t.value == 3 and t.viability == 3 and t.error_cost <= 1:   # rule 4
        level = 4; reasons.append("complexity, value, viability all 3 and error cost <= 1 -> promoted to L4")
    cap = {3: 2, 2: 3}.get(t.error_cost, 4)   # rule 5
    if level > cap:
        level = cap; reasons.append(f"error_cost={t.error_cost} -> capped at L{cap}")
    approval = t.error_cost >= 2 and level >= 2
    if approval: reasons.append("error_cost >= 2 -> human approval on write actions")
    return Recommendation(level, "build", approval, reasons)

r = recommend(Task("autonomous payment wiring", complexity=3, value=3, viability=3, error_cost=3))
print(r.label, "| approval:", r.human_approval); print(*r.reasons, sep="\n  - ")

### Test the helper like any other code
Hand-picked cases pin down the *intent*; a **property test** over the whole 4⁴ = 256 input grid catches rule interactions you did not think of.
The key property for a safety rubric is **monotonicity in risk**: raising the cost of error must never *raise* the autonomy level.

In [ ]:
import itertools

# (1) example-based tests
assert recommend(Task("summary", 0, 1, 3, 0)).level == 0
assert recommend(Task("triage", 1, 2, 3, 1)).level == 1
assert recommend(Task("deep research", 3, 3, 3, 1)).level == 4
assert recommend(Task("payments", 3, 3, 3, 3)).level == 2           # error-cost cap beats everything
assert recommend(Task("vendor negotiation", 3, 3, 1, 3)).verdict == "not_yet"
assert recommend(Task("trivial but messy", 3, 0, 3, 0)).level == 1  # value cap

# (2) property tests over the full grid
grid = [Task("g", *vals) for vals in itertools.product(range(4), repeat=4)]
assert len(grid) == 256
for t in grid:
    base = recommend(t)
    assert 0 <= base.level <= 4
    if t.error_cost < 3:   # monotone in error cost
        riskier = recommend(Task("g", t.complexity, t.value, t.viability, t.error_cost + 1))
        assert riskier.level <= base.level, (t, base.level, riskier.level)
    if t.viability >= 2 and t.error_cost >= 2 and base.level >= 2:
        assert base.human_approval
    if t.viability <= 1:
        assert base.verdict == "not_yet"
print("all example + property tests pass (256 grid points)")

## 3 · Use it: eight real tasks from the retailer
Each row is a request from a different team. The scores are *judgements*, so we write the reasoning next to them, which is
the part reviewers will challenge. Then we check that the helper lands where a thoughtful engineer would.

| Task | Why these scores |
|---|---|
| Summarise weekly sales call transcripts | one step, harmless, we can eyeball it |
| Route + draft replies to inbound tickets | fixed steps (classify → draft); wrong draft is reviewed by a human |
| Invoice capture → validate → post to ERP | fixed steps but a wrong posting costs money to unwind |
| Order-support: look up order, check policy, refund | branching path (which tools, how many?); refunds are real money but capped & reversible |
| Fix a failing unit test in the monorepo | open-ended search, but a git branch + CI makes mistakes cheap |
| Competitor landscape research report | exploration is the point; high value; wrong claim is embarrassing, not catastrophic |
| Auto-approve consumer loan applications | some branching; regulated, hard to reverse; model reliability on this data unproven |
| Autonomously negotiate supplier contracts | open-ended + irreversible, and no tool or evidence of capability yet |

In [ ]:
TASKS = [
    #     name,                                    cx val via err   expected level, expected verdict
    (Task("Summarise sales-call transcripts",       0, 1, 3, 0), 0, "build"),
    (Task("Route + draft replies to tickets",       1, 2, 3, 1), 1, "build"),
    (Task("Invoice capture -> ERP posting",         1, 2, 3, 2), 1, "build"),
    (Task("Order support with refunds",             2, 2, 3, 2), 2, "build"),
    (Task("Fix failing unit test in monorepo",      3, 2, 3, 1), 3, "build"),
    (Task("Competitor landscape research report",   3, 3, 3, 1), 4, "build"),
    (Task("Auto-approve consumer loans",            2, 3, 2, 3), 2, "build"),
    (Task("Autonomously negotiate supplier deals",  3, 3, 1, 3), 0, "not_yet"),
]
print(f"{'task':<42}{'level':<22}{'verdict':<9}{'approval'}")
for t, exp_level, exp_verdict in TASKS:
    r = recommend(t)
    print(f"{t.name:<42}{r.label:<22}{r.verdict:<9}{r.human_approval}")
    assert (r.level, r.verdict) == (exp_level, exp_verdict), (t.name, r)
n_agents = sum(1 for t, *_ in TASKS if recommend(t).level >= 2 and recommend(t).verdict == "build")
print(f"\n{n_agents}/8 requests genuinely need an agent; the rest are cheaper, more predictable workflows or single calls.")
assert n_agents == 4

### Which question matters most? A sensitivity check
A rubric you can't probe is a rubric you can't trust. For each task, nudge one score by ±1 and count how often the *recommended level* changes.
If one question dominates everything, either that is intended (error cost, as a safety cap) or the rubric is badly shaped.

In [ ]:
import collections
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
flips = collections.Counter(); total = collections.Counter()
for t, *_ in TASKS:
    base = recommend(t)
    for f in ("complexity", "value", "viability", "error_cost"):
        for d in (-1, +1):
            v = getattr(t, f) + d
            if 0 <= v <= 3:
                kw = {k: getattr(t, k) for k in ("complexity", "value", "viability", "error_cost")}; kw[f] = v
                r2 = recommend(Task(t.name, **kw))
                total[f] += 1; flips[f] += int((r2.level, r2.verdict) != (base.level, base.verdict))
for f in total: print(f"{f:<11} changes the outcome in {flips[f]:>2}/{total[f]:<2} single-step nudges ({flips[f]/total[f]:.0%})")
assert flips["complexity"] > 0 and flips["viability"] > 0     # both matter
assert flips["value"] < flips["complexity"]                   # value only caps; complexity sets the level

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.8))
colors = {0: "#999999", 1: "#4c78a8", 2: "#f58518", 3: "#e45756", 4: "#72b7b2"}
for t, *_ in TASKS:
    r = recommend(t)
    ax.scatter(t.complexity + rng.uniform(-.12, .12), t.error_cost + rng.uniform(-.12, .12), s=90, color=colors[r.level], edgecolor="k", zorder=3)
for lv, c in colors.items(): ax.scatter([], [], color=c, edgecolor="k", label=LADDER[lv])
ax.set_xlabel("complexity (unpredictability of path)"); ax.set_ylabel("cost of error"); ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_title("Where the 8 requests land"); ax.legend(fontsize=7, loc="upper left"); ax.grid(alpha=.3)
plt.show()

## 4 · Workflow vs agent, side by side
Same domain, same two tickets, two architectures. Watch **who owns the control flow** and what that does to the number of model calls.

* **Workflow:** code calls the model exactly twice, always: *classify → (code) look up order → draft*. The shape never changes.
* **Agent:** one loop; the model chooses which tools to call and when to stop.

Ticket A is easy ("where is order 1043?"). Ticket B is messier (needs the order *and* the policy). In OFFLINE mode the model is a script,
but the **call counts are real consequences of the control flow**, not something we typed.

In [ ]:
from agentkit import tool, ToolBox, get_llm, say, call, run_agent

ORDERS = {"1042": {"item": "Trail Shoes", "total": 129.0, "status": "delivered", "days_since_delivery": 3},
          "1043": {"item": "Rain Jacket", "total": 219.0, "status": "in_transit", "days_since_delivery": None}}
POLICY = "Refunds: within 30 days of delivery, delivered orders only, up to order total."

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by its numeric ID.

    Args:
        order_id: Numeric order ID such as "1042".
    """
    if order_id not in ORDERS: raise KeyError(f"No order {order_id}")
    return ORDERS[order_id]

@tool
def get_refund_policy() -> str:
    """Return the current refund policy text."""
    return POLICY

TB = ToolBox(lookup_order, get_refund_policy)

# --- Workflow: the control flow is Python ---------------------------------------------------
def workflow(llm, ticket: str, order_id: str) -> dict:
    n_llm = 0
    cat = llm.chat([{"role": "user", "content": f"Classify as order_status|refund|other: {ticket}"}]); n_llm += 1
    order, _ = TB.run("lookup_order", {"order_id": order_id})                     # code decides to call this, always
    draft = llm.chat([{"role": "user", "content": f"Category {cat.text}. Order {order}. Draft a reply to: {ticket}"}]); n_llm += 1
    return {"reply": draft.text, "llm_calls": n_llm, "category": cat.text}

# --- Agent: the control flow is the model ---------------------------------------------------
def agent(llm, ticket: str):
    return run_agent(llm, TB, ticket, system="You are a support agent. Use tools for facts; answer briefly.", max_turns=6)

tA = "Where is order 1043?"
tB = "Order 1042 shoes don't fit - can I get my money back?"

wf_A = workflow(get_llm(script=[say("order_status"), say("Order 1043 is in transit.")]), tA, "1043")
wf_B = workflow(get_llm(script=[say("refund"), say("Order 1042 is eligible; we'll refund $129.")]), tB, "1042")

ag_A = agent(get_llm(script=[call("lookup_order", order_id="1043"), say("Order 1043 (Rain Jacket) is in transit.")]), tA)
ag_B = agent(get_llm(script=[call("lookup_order", order_id="1042"), call("get_refund_policy"),
                             say("Order 1042 was delivered 3 days ago, within the 30-day policy, so it is eligible for a $129 refund.")]), tB)

print(f"{'':<10}{'ticket A (easy)':<22}{'ticket B (policy needed)'}")
print(f"{'workflow':<10}{wf_A['llm_calls']} LLM calls, 1 tool{'':<8}{wf_B['llm_calls']} LLM calls, 1 tool (no policy lookup!)")
print(f"{'agent':<10}{ag_A.turns} LLM calls, {ag_A.tracer.n_tool_calls()} tool{'':<9}{ag_B.turns} LLM calls, {ag_B.tracer.n_tool_calls()} tools")
print("\nagent trace for ticket B:"); ag_B.tracer.show()
assert wf_A["llm_calls"] == wf_B["llm_calls"] == 2        # fixed shape: identical for every input
assert ag_A.turns != ag_B.turns                           # the agent's shape depends on the input
assert ag_B.tracer.n_tool_calls("get_refund_policy") == 1 and ag_A.tracer.n_tool_calls("get_refund_policy") == 0

**What to notice.** The workflow gave ticket B an answer *without ever consulting the policy*, because nobody wired that step into the
pipeline. The agent fetched it because the model judged it necessary. That is the flexibility you pay for: the agent's cost and latency now
vary per input (2 vs 3 model calls here, dozens in real tasks) and you can no longer say in advance what it will do.
Neither is "better": for tickets that *never* need the policy, the workflow is the right tool.

## 5 · Break it: a tour of agent failure modes
Agents fail in ways a single call does not, because errors **compound** and context **accumulates**. Two failures are structural,
so you can compute them before building anything.

| Failure mode | What happens | First guard (lesson) |
|---|---|---|
| Compounding error | a 95 %-reliable step done 20× in sequence succeeds ≈ 36 % of the time | fewer steps, verification, checkpoints (AG04, AG14) |
| Runaway loop | model repeats a failing call forever | `max_turns`, budgets (AG02) |
| Wrong tool / bad args | ambiguous tools get misused | tool design (AG03) |
| Context rot / cost growth | every turn re-sends all prior turns | compaction, notes (AG05) |
| Hallucinated success (5c) | "Done!" with no evidence | verify outcomes with code (AG04, AG14) |
| Over-reach / injection | acts beyond its mandate, obeys text in a web page | permissions, sandboxing (AG09) |

### 5a · Compounding error: a Monte-Carlo you can check by algebra
If each step succeeds independently with probability *p*, a *n*-step run succeeds with *pⁿ*. Let's simulate and compare with the formula.

In [ ]:
rng = np.random.default_rng(0)
ns = np.arange(1, 31)
fig, ax = plt.subplots(figsize=(6, 3.4))
for p in (0.90, 0.95, 0.99):
    analytic = p ** ns
    sim = np.array([(rng.random((20000, n)) < p).all(axis=1).mean() for n in ns])   # 20k simulated runs per n
    assert np.abs(sim - analytic).max() < 0.02                                       # simulation agrees with p**n
    ax.plot(ns, analytic, label=f"p={p}")
ax.set_xlabel("steps in the run (n)"); ax.set_ylabel("P(all steps succeed)"); ax.set_title("Reliability decays geometrically"); ax.legend(); ax.grid(alpha=.3)
plt.show()
print(f"p=0.95, n=20  -> {0.95**20:.2f}   |   p=0.99, n=20 -> {0.99**20:.2f}")
assert 0.95 ** 20 < 0.40 and 0.99 ** 20 > 0.80

*Implication.* To keep a long run reliable you either **raise per-step reliability** (better tools, verification after risky steps) or
**shorten the run** (decompose, checkpoint). Real steps aren't independent (a good verifier can *recover* from errors), but the geometric
decay is the default you must design against. This is why AG04 adds verification steps and AG14 measures per-step success.

### 5b · Cost growth: every turn re-sends the whole conversation
A turn's input is the system prompt plus all earlier turns. If each turn adds *k* tokens, input tokens over *n* turns sum to ≈ k·n(n+1)/2 — **quadratic**.
(Prompt caching, AG05/A09, discounts re-sent tokens but doesn't remove the growth in context.)

In [ ]:
from agentkit.trace import PRICES
from agentkit.llm import DEFAULT_MODEL

def run_cost(n_turns: int, tokens_per_turn: int = 800, system_tokens: int = 1500, out_per_turn: int = 150) -> float:
    pin, pout = PRICES[DEFAULT_MODEL]
    inp = sum(system_tokens + tokens_per_turn * t for t in range(n_turns))   # turn t re-sends t earlier turns
    return (inp * pin + n_turns * out_per_turn * pout) / 1e6

c5, c10, c20 = run_cost(5), run_cost(10), run_cost(20)
print(f"5 turns ${c5:.4f} | 10 turns ${c10:.4f} | 20 turns ${c20:.4f}")
print(f"doubling turns (10 -> 20) multiplies cost by {c20 / c10:.2f}x, not 2x")
assert c20 / c10 > 2.5      # super-linear

### 5c · Hallucinated success: trust the ledger, not the narration
Agents report what they *believe* happened. Here the scripted model says "refund issued" **without ever calling the refund tool**
(a real failure mode, usually triggered by long contexts or by tools that errored earlier). The only defence is checking *state*, not text.

In [ ]:
LEDGER = []   # the real system of record

@tool
def issue_refund(order_id: str, amount: float) -> str:
    """Issue a refund for a delivered order.

    Args:
        order_id: Numeric order ID.
        amount: Amount in NZD, up to the order total.
    """
    LEDGER.append((order_id, amount)); return f"Refund of ${amount:.2f} issued for {order_id}."

TB2 = ToolBox(lookup_order, issue_refund)
llm = get_llm(script=[call("lookup_order", order_id="1042"),
                      say("All sorted! I've refunded $129.00 to your original payment method.")])   # never called issue_refund
res = run_agent(llm, TB2, "Refund order 1042 please.", system="Support agent.", max_turns=4)

claimed = "refund" in res.final_text.lower() and "$" in res.final_text
actually = any(oid == "1042" for oid, _ in LEDGER)
print("agent claims refund:", claimed, "| ledger shows refund:", actually)
print("tool calls made:", [e.name for e in res.tracer.events if e.kind == "tool"])
assert claimed and not actually, "the narration and the system of record disagree: that is the bug"
verified = claimed and actually
print("outcome verified:", verified, "-> do NOT tell the customer 'done'")

## 6 · Exercises
**Practice (30 min)**
1. Add two tasks of your own from a workplace you know (scored with reasons) to `TASKS`. Is there one where you *disagree* with the helper? Write down which rule you would change.
2. In the compounding-error cell, find the smallest per-step reliability *p* such that a 15-step run succeeds ≥ 90 % of the time. Assert it analytically.

**Stretch (60 min)**
3. Add a rule 6, **latency budget**: a task flagged `interactive=True` (user waiting) may not exceed L2. Add it without breaking the 256-point property tests (extend the Task dataclass with a default).
4. Allow *partial verification* in the reliability model: after each step a check catches an error with probability *q* and retries once. Derive the new per-step success and re-plot.

**Challenge (2 h)**
5. Take a workflow you built earlier (e.g. the A01 invoice extractor). Write the strongest case that it should be an agent, then the strongest case that it should stay a workflow. Decide using only evidence you could measure (call counts, failure rate, cost).

In [ ]:
# Starter for exercise 2: smallest p with p**15 >= 0.90
def min_step_reliability(n_steps: int, target: float) -> float:
    # TODO: closed form p = target ** (1 / n_steps)
    raise NotImplementedError

# p = min_step_reliability(15, 0.90)
# assert abs(p ** 15 - 0.90) < 1e-9 and 0.99 < p < 0.995

# Starter for exercise 3: uncomment and extend Task with `interactive: bool = False`
# def test_interactive_cap():
#     t = Task("live chat", 3, 3, 3, 0)
#     ...  # assert recommend(...).level <= 2 when interactive=True

## ✅ Checkpoint (answer before moving on)
1. Your colleague says "our pipeline classifies, retrieves, then drafts. It's an agent because it uses an LLM and tools." Do you agree? Why / why not?
2. A task has complexity 2, value 3, viability 3 and error cost 3. What level does the rubric give, and what extra control does it require?
3. Why is "viability" a *gate* rather than just another weighted score?
4. A 12-step agent run has 97 % per-step reliability. Roughly what fraction of runs succeed, and name two ways to improve it that don't involve a better model.

<details><summary>Answers</summary>

1. No: the control flow is fixed in code, so it is a **workflow** (L1). It becomes an agent only if the *model* decides which tools to call and when to stop.
2. Level 2 (error cost 3 caps at L2) with **human approval on write actions** (error cost ≥ 2).
3. A weighted sum lets a high score on value or complexity compensate for tools that don't exist or a model that can't do the job. No amount of value makes a non-viable system work; so it must veto.
4. 0.97¹² ≈ 0.69. Improve by shortening the run (decompose / checkpoint) and by adding verification after risky steps, or by making tools harder to misuse (AG03).
</details>

## 📌 Key takeaways
- **Workflow = your code owns control flow; agent = the model does.** Most production value sits at L1–L2.
- Choose the **lowest autonomy level that solves the task**; each rung costs predictability, money and debuggability.
- Use the **4-question test** (complexity, value, viability, cost of error) and put it in *code with tests*, so decisions are reviewable.
- Reliability decays as *pⁿ* and cost grows super-linearly with turns, so design for **short runs, verification and budgets**.
- Error cost caps autonomy: the more irreversible the action, the more a human stays in the loop.

## 📚 Further reading
- Anthropic Engineering, *Building effective agents* (workflows vs agents, the augmented LLM, when to use agents) — anthropic.com/engineering/building-effective-agents
- Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models* (2022), arXiv:2210.03629
- Anthropic docs: *Tool use with Claude* — docs.anthropic.com

**Next → AG02 · The Agent Loop from Scratch** (build the loop you just decided to use), then **AG03 · Tools Done Right**.